# Single-Country Endowment Economy Baseline

This notebook simulates the **single-country endowment OLG economy** that underlies
the two-country bubble model in Hirano et al. (2025).

### Model in one paragraph
A young agent receives endowment $e_t$ and saves $A_t = \beta e_t$ in the Lucas tree (stock).
Market clearing trivially gives $Q_t = \beta e_t$.  The economy switches between an
**unbalanced-growth u-state** (growth rate $g_{e,u}$, bubble persists) and an absorbing
**balanced b-state** (growth rate $g_{e,b}$, bubble pops) with transition probability
$1-\pi_{\text{persist}}$.  At the switch date $Q^b = \beta \lambda_e e^u$,
$D^b = \lambda_D D^u$ (Assumption 5).

Because $\omega = 1$ and $\theta = 0$ trivially (single asset, single agent),
the entire equilibrium is **closed-form** — no solver needed.

In [ ]:
include("SingleCountryOLG.jl")
using Plots, Printf
default(fontfamily="Computer Modern", linewidth=2, framestyle=:box, legend=:topright)

## 1. Baseline simulation

Run with default parameters (matching the two-country setup: $\beta=0.5$, $\gamma=0.5$,
$\pi=0.70$, $g_{e,u}=1.025^5$, $g_{e,b}=1.01^5$, $g_{D,u}=1.015^5$, $\lambda_e=\lambda_D=1$).

In [ ]:
p = SCParams()
r = run_sc_simulation(p; verbose=true)

## 2. Exogenous paths: endowment and dividends

In [ ]:
ts = 1:p.T_max
e  = [r.states[t].e for t in ts]
D  = [r.states[t].D for t in ts]
d  = [r.states[t].d for t in ts]  # D/e ratio

p1 = plot(ts, e, label="Endowment e_t", yscale=:log10,
          title="Exogenous paths (log scale)",
          xlabel="Period t", ylabel="Level")
plot!(p1, ts, D, label="Dividends D_t", linestyle=:dash)

p2 = plot(ts, d, label="d_t = D_t/e_t",
          title="Dividend-endowment ratio",
          xlabel="Period t", ylabel="D/e")
hline!(p2, [0.0], color=:black, linestyle=:dot, label="0")

plot(p1, p2, layout=(1,2), size=(900,350))

**Key observation**: $d_t = D_t/e_t$ declines geometrically at rate
$g_{D,u}/g_{e,u} = (1.015/1.025)^5 \approx 0.952$ per period.
This ensures **Condition 2a** ($\sum d_t < \infty$) is satisfied.

## 3. Returns and consumption ratio

In [ ]:
ts2 = 2:p.T_max
R_u = [r.states[t].R_u for t in ts2]
R_b = [r.states[t].R_b for t in ts2]
cr  = [r.diagnostics.consump_ratio[t] for t in ts2]

p3 = plot(ts2, R_u, label="R^u_t (u-continuation)",
          title="Portfolio returns",
          xlabel="Period t", ylabel="Gross return")
plot!(p3, ts2, R_b, label="R^b_t (b-switch)", linestyle=:dash)
hline!(p3, [p.g_e_u], color=:grey, linestyle=:dot, label="g_e_u")
hline!(p3, [p.g_e_u * p.λ_e], color=:grey, linestyle=:dashdot, label="g_e_u·λ_e")

p4 = plot(ts2, cr, label="C^b_t / C^u_t",
          title="Consumption ratio (b vs u state)",
          xlabel="Period t", ylabel="Ratio")
hline!(p4, [p.λ_e], color=:red, linestyle=:dash, label="λ_e (asymptote)")
hline!(p4, [1.0], color=:black, linestyle=:dot, label="1")

plot(p3, p4, layout=(1,2), size=(900,350))

**Key observation**: Both returns converge to $g_{e,u}$ (resp. $g_{e,u}\lambda_e$) as $d_t \to 0$.
The ratio $C^b/C^u = R^b/R^u \to \lambda_e$ — a **constant**, not declining to 0.

## 4. Bubble diagnostic conditions

In [ ]:
sum2a = r.diagnostics.sum_dividend_ratio
sum2b = r.diagnostics.sum_cond_2b
terms = r.diagnostics.cond_2b_terms

p5 = plot(ts, sum2a[ts], label="Cumulative Σ d_t  (Cond. 2a)",
          title="Condition 2a: Σ D^u_t / e^u_t",
          xlabel="Period t", ylabel="Cumulative sum")

p6 = plot(ts2, sum2b[ts2], label="Cumulative Σ (C^b/C^u)^{1-γ}  (Cond. 2b)",
          title="Condition 2b: Σ (C^b/C^u)^{1-γ}",
          xlabel="Period t", ylabel="Cumulative sum")
# Reference: linear growth
λγ = p.λ_e^(1 - p.γ)
plot!(p6, ts2, λγ .* (ts2 .- 1), color=:grey, linestyle=:dash,
      label=@sprintf("λ_e^{1-γ}·t = %.3f·t (asymptote)", λγ))

plot(p5, p6, layout=(1,2), size=(900,350))

**Interpretation**:
- **Condition 2a** (left): the cumulative sum plateaus — it converges. ✓
- **Condition 2b** (right): the cumulative sum tracks the linear reference $\lambda_e^{1-\gamma}\cdot t$ — it **diverges**. ✗

Since condition 2b fails, the bubble does **not** exist in the single-country baseline.

## 5. Sensitivity to $\lambda_e$

Varying $\lambda_e \in \{1.0, 0.9, 0.8, 0.5\}$ shifts $C^b/C^u$, but condition 2b still diverges
because the terms converge to a **positive constant** $(\lambda_e)^{1-\gamma}$.

In [ ]:
λe_vals = [1.0, 0.9, 0.8, 0.5]
colors   = [:steelblue, :darkorange, :forestgreen, :crimson]

p7 = plot(title="Condition 2b for different λ_e",
          xlabel="Period t", ylabel="Cumulative Σ (C^b/C^u)^{1-γ}",
          legend=:topleft)
p8 = plot(title="C^b/C^u ratio for different λ_e",
          xlabel="Period t", ylabel="C^b_t / C^u_t",
          legend=:bottomright)

for (λe, col) in zip(λe_vals, colors)
    ri  = run_sc_simulation(SCParams(λ_e=λe); verbose=false)
    s2b = ri.diagnostics.sum_cond_2b
    cr_i = ri.diagnostics.consump_ratio
    lbl = @sprintf("λ_e = %.1f", λe)
    plot!(p7, ts2, s2b[ts2], label=lbl, color=col)
    plot!(p8, ts2, cr_i[ts2], label=lbl, color=col)
end
hline!(p8, [0.0], color=:black, linestyle=:dot, label="0")

plot(p7, p8, layout=(1,2), size=(900,350))

**Key finding**: Even with $\lambda_e = 0.5$ (50\% endowment drop at bubble pop),
$C^b/C^u \to 0.5$ (constant), so $(C^b/C^u)^{1-\gamma} \approx 0.71$ — still a constant term.
The cumulative sum grows linearly at rate $(\lambda_e)^{1-\gamma}$ in all cases.

## 6. Why the two-country model is different

The table below compares the mechanisms in each model:

| Feature | Single-country | Two-country |
|---|---|---|
| Portfolio weight on US stock | $\omega = 1$ (trivial) | $\omega^u \approx 0.14$, $\omega^b \approx 0.80$ |
| US stock price $Q^u_{US}$ | $\beta e_t$ (= total savings) | $\omega^u \cdot S + \omega^*_u \cdot S^* \ll \beta e_t$ |
| US stock price $Q^b_{US}$ | $\beta \lambda_e e_t$ | $\omega^b \cdot S + \omega^*_b \cdot S^* \gg Q^u_{US}$ |
| $C^b / C^u$ | $\to \lambda_e$ (constant) | Can vary; driven by $Q^b_{US} / Q^u_{US}$ |
| Condition 2b | Always diverges | Potentially converges |
| Bubble exists | Never (with standard params) | Yes (if conditions met) |

The **cross-country portfolio rebalancing** ($\omega^u \ll \omega^b$) depresses $Q^u_{US}$
relative to $Q^b_{US}$, which is the core mechanism that can make $C^b/C^u$ deviate
from a constant and allow condition 2b to converge.

In [ ]:
# Summary table
println("Single-country baseline summary")
println("═" ^ 50)
@printf("  Initial d_0 = D_0/e_0:    %.4f\n", r.states[1].d)
@printf("  Final   d_T = D_T/e_T:    %.2e\n", r.states[end].d)
@printf("  C^b/C^u at t=2:           %.6f\n", r.diagnostics.consump_ratio[2])
@printf("  C^b/C^u at t=T:           %.6f\n", r.diagnostics.consump_ratio[end])
@printf("  λ_e^{1-γ} (asymptote):    %.6f\n", p.λ_e^(1-p.γ))
@printf("  Σ d_t (condition 2a):     %.6f  → converges: %s\n",
        r.diagnostics.sum_dividend_ratio[end], r.diagnostics.condition_2a_converges)
@printf("  Σ (C^b/C^u)^{1-γ} (2b):  %.6f  → converges: %s\n",
        r.diagnostics.sum_cond_2b[end], r.diagnostics.condition_2b_converges)
@printf("  Bubble exists:            %s\n", r.diagnostics.bubble_exists)